Stratified K-Fold

a volte solo k-fold non basta perchè i dati NON sono sempre distribuiti bene. 
Esempio ho 1.000 record 950 classe 0 e 50 classe 1. Se faccio k-fold 'normale' può succedere che alcuni fold non hanno valori per la classe 1, quindi il modello risulta instabile e metriche falsate.
Con Stratified k-fold si mantiene la stessa proporzione della classi in ogni fold, quindi ogni fold è una 'mini copia' del dataset orginale.

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
data=load_iris()
X=pd.DataFrame(data.data, columns=data.feature_names)
y=pd.Series(data.target,name="Species")

df=X.copy()
df["species"]=y
display(df)
print (df.groupby("species").mean())
print (df.groupby("species").mean())
df["species"].value_counts()  #ho anche valori nulli
df.groupby("species").count()  #conta i valori non nulli per colonna

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


         sepal length (cm)  sepal width (cm)  petal length (cm)  \
species                                                           
0                    5.006             3.428              1.462   
1                    5.936             2.770              4.260   
2                    6.588             2.974              5.552   

         petal width (cm)  
species                    
0                   0.246  
1                   1.326  
2                   2.026  
         sepal length (cm)  sepal width (cm)  petal length (cm)  \
species                                                           
0                    5.006             3.428              1.462   
1                    5.936             2.770              4.260   
2                    6.588             2.974              5.552   

         petal width (cm)  
species                    
0                   0.246  
1                   1.326  
2                   2.026  


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
species,,,,
0,50,50,50,50
1,50,50,50,50
2,50,50,50,50


istanziamo il modello della regressione logistica

In [8]:
model=LogisticRegression(solver="lbfgs",random_state=42) #lbsgs algoritmo che supportaa clasicazione multiclasse

In [ ]:
k=10  #n.di fold (normalmente si mette o 5 o 10), questo dataset è piccolo sia per righe sia per colonne, pertanto 10 va bene

calcoli sul k-fold

In [ ]:
accuracy_standard=[]

kf_standard=KFold(n_splits=k,shuffle=True,random_state=42)

for train_index, test_index in kf_standard.split(X):  #ritorna gli indici di come ha fatto lo split, indici di train e test
    #prendo il dataset originale e seleziono le righe corrispondenti agli indici
    X_train, X_test=X.iloc[train_index],X.iloc[test_index]
    y_train, y_test=y.iloc[train_index],y.iloc[test_index]
    #standarizzo le feature (da fare necessariamente dopo lo split)
    scaler=StandardScaler()
    X_train_scaled=scaler.fit_transform(X_train)
    X_test_scaled=scaler.fit_transform(X_test)
    #alleno il modello
    model.fit(X_train_scaled,y_train)
    #predizioni
    y_pred_standard=model.predict(X_test_scaled)
    #calcolo le metriche di predizione
    score=model.score(X_test_scaled,y_test) #calcolo accuracy (% di predizioni corrette)
    #salvo le metriche
    accuracy_standard.append(score)


calcoli sul StatifiedKFold

In [29]:
accuracy_stratified=[]

kf_stratified=StratifiedKFold(n_splits=k,shuffle=True,random_state=42)

for train_index, test_index in kf_stratified.split(X,y):  #ritorna gli indici di come ha fatto lo split, indici di train e test
    #prendo il dataset originale e seleziono le righe corrispondenti agli indici
    X_train, X_test=X.iloc[train_index],X.iloc[test_index]
    y_train, y_test=y.iloc[train_index],y.iloc[test_index]
    #standarizzo le feature (da fare necessariamente dopo lo split)
    scaler=StandardScaler()
    X_train_scaled=scaler.fit_transform(X_train)
    X_test_scaled=scaler.transform(X_test)
    #alleno il modello
    model.fit(X_train_scaled,y_train)
    #predizioni
    y_pred_standard=model.predict(X_test_scaled)
    #calcolo le metriche di predizione
    score=model.score(X_test_scaled,y_test) #calcolo accuracy (% di predizioni corrette)
    #salvo le metriche
    accuracy_stratified.append(score)

vediamo i risultati ottenuti

In [28]:
#calcolo la media delle accuracy
mean_standard=np.mean(accuracy_standard)
std_standard=np.std(accuracy_standard)
mean_stratified=np.mean(accuracy_stratified)
std_stratifies=np.std(accuracy_stratified)

print(F"K=Fold Stanrdard:\n\tMedia:\t{mean_standard:.4f}\n\tSTD:\t{std_standard:.4f}\n\t{np.round(accuracy_standard,4)}")
print(F"K=Stratified:\n\tMedia:\t{mean_stratified:.4f}\n\tSTD:\t{std_stratifies:.4f}\n\t{np.round(accuracy_stratified,4)}")

K=Fold Stanrdard:
	Media:	0.8933
	STD:	0.0854
	[0.9333 0.9333 0.7333 0.9333 1.     0.8    0.8667 0.9333 1.     0.8   ]
K=Stratified:
	Media:	0.9533
	STD:	0.0670
	[1.     1.     1.     1.     0.8    0.9333 1.     1.     0.9333 0.8667]


Accuracy è più alta in K-Fold Stratified che non in K-Fold Standard. Questo è in linea con la teoria
Guardando gli array delle accuracy di K-Fold Standard abbiamo un range più alto (0.733 a 1)
Guardando gli array delle accuracy di K-Fold Stratified abbiamo un range più contenuto (0.8 a 1)
Questo perchè K-Fold non garantisce distribuzione uniforme nei fold, quindi l'accuracy può cambiare molto da un fold all'altro.
Con Standard alcuni fold hanno meno esempi di una classe, pertanto il modello peggiora su queste classi.
Con Stratified invece le proporzioni sono mantenute ed il modello è sempre 'allenato bene'

Conclusione:
Stratified K-Fold fornisce una stima più affidabile delle performance del modello (0.9533) rispetto al K-Fold standard (0.8933), in quanto preserva la distribuzione delle classi nei vari fold, riducendo la variabilità dei risultati.



Il modello valutato con Stratified K-Fold mostra una performance media superiore (0.9533 vs 0.8933) e una minore variabilità (std 0.0670 vs 0.0854) rispetto al K-Fold standard. Questo indica che la stratificazione consente una distribuzione più uniforme delle classi nei vari fold, migliorando sia la stabilità che l’affidabilità della valutazione del modello.

L’utilizzo dello Stratified K-Fold evidenzia una stima più robusta delle performance, riducendo la variabilità osservata nel K-Fold standard. Tuttavia, l’analisi basata sulla sola accuracy è limitata e dovrebbe essere integrata con metriche per classe, soprattutto se il dataset presenta squilibri.
Si potrebbe utilizzare, per ogni modello:

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred)) 

Facciamo un plot di confronto tra le accuracy